# Ćwiczenia 2: Model ML, FastAPI i serwowanie predykcji

*Studia zaoczne | 1,5h*

## Cel

- Wytrenowanie modelu detekcji fraudów,
- Serwowanie modelu przez FastAPI,
- Podłączenie do Kafki — scoring w czasie rzeczywistym.

---

## Część 1: Trening modelu (25 min)

### Zadanie 1.1 — Przygotuj dane i wytrenuj model

In [11]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pickle

np.random.seed(42)

# Normalne transakcje
normal = pd.DataFrame({
    'amount': np.random.lognormal(5, 1, 2000).clip(5, 5000),
    'hour': np.random.normal(14, 4, 2000).clip(0, 23).astype(int),
    'is_electronics': np.random.binomial(1, 0.3, 2000),
    'tx_per_day': np.random.poisson(3, 2000),
    'fraud': 0
})

# Fraudy
fraud = pd.DataFrame({
    'amount': np.random.uniform(500, 6000, 100),  # mniej oczywiste
    'hour': np.random.randint(0, 24, 100),        # nie tylko noc
    'is_electronics': np.random.binomial(1, 0.5, 100),
    'tx_per_day': np.random.poisson(5, 100),
    'fraud': 1
})

X['noise'] = np.random.normal(0, 1, len(X))

df = pd.concat([normal, fraud], ignore_index=True).sample(frac=1, random_state=42)
print(f"Dataset: {len(df)} wierszy, fraud rate: {df['fraud'].mean():.1%}")

Dataset: 2100 wierszy, fraud rate: 4.8%


### Zadanie 1.2 — Podziel dane, wytrenuj, oceń

In [7]:
features = ['amount', 'hour', 'is_electronics', 'tx_per_day']
X = df[features]
y = df['fraud']

# 1. train_test_split (80/20, stratify=y)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 2. RandomForestClassifier(100)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predykcje
y_pred = model.predict(X_test)

# 3. classification_report
print(classification_report(y_test, y_pred))

# 4. zapis modelu
with open('fraud_model.pkl', 'wb') as f:
    pickle.dump(model, f)

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       400
           1       0.80      0.80      0.80        20

    accuracy                           0.98       420
   macro avg       0.90      0.90      0.90       420
weighted avg       0.98      0.98      0.98       420



---

## Część 2: FastAPI (25 min)

### Zadanie 2.1 — Utwórz API serwujące model

In [8]:
%%file fraud_api.py
from fastapi import FastAPI
from pydantic import BaseModel
import pickle, numpy as np

app = FastAPI(title="Fraud Detection API")
model = pickle.load(open('fraud_model.pkl', 'rb'))

class Transaction(BaseModel):
    amount: float
    hour: int
    is_electronics: int
    tx_per_day: int


# Endpoint POST /score
@app.post("/score")
def score(tx: Transaction):
    # zamiana na array
    data = np.array([[ 
        tx.amount, 
        tx.hour, 
        tx.is_electronics, 
        tx.tx_per_day 
    ]])
    
    # predykcja
    pred = model.predict(data)[0]
    
    # prawdopodobieństwo klasy 1 (fraud)
    proba = model.predict_proba(data)[0][1]
    
    return {
        "is_fraud": bool(pred),
        "fraud_probability": float(proba)
    }

Writing fraud_api.py


Uruchom w terminalu:
```bash
uvicorn fraud_api:app --host 0.0.0.0 --port 8001
```

### Zadanie 2.2 — Przetestuj API

In [9]:
import requests

# Test normalna
r = requests.post("http://localhost:8001/score",
    json={"amount": 150, "hour": 14, "is_electronics": 0, "tx_per_day": 3})
print("Normalna:", r.json())

# TWÓJ KOD — przetestuj podejrzaną transakcję (amount=5500, hour=3, is_electronics=1, tx_per_day=12)
# Test podejrzana
r = requests.post("http://localhost:8001/score",
    json={"amount": 5500, "hour": 3, "is_electronics": 1, "tx_per_day": 12})
print("Podejrzana:", r.json())

Normalna: {'is_fraud': False, 'fraud_probability': 0.0}
Podejrzana: {'is_fraud': True, 'fraud_probability': 0.96}


---

## Część 3: Kafka + ML (25 min)

### Zadanie 3.1 — Konsument z scoringiem ML

Napisz konsumenta, który czyta z `transactions`, odpytuje API i wysyła alerty.

In [10]:
%%file ml_consumer.py
from kafka import KafkaConsumer, KafkaProducer
from datetime import datetime
import json, requests

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='ml-scoring',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

alert_producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

API_URL = "http://localhost:8001/score"

for message in consumer:
    tx = message.value
    
    try:
        # 1. Wyciągnięcie cech
        amount = tx.get("amount", 0)

        # timestamp → hour
        timestamp = tx.get("timestamp")
        hour = datetime.fromisoformat(timestamp).hour if timestamp else 0

        is_electronics = tx.get("is_electronics", 0)
        
        # brak tej cechy w streamie → stała wartość
        tx_per_day = 5

        features = {
            "amount": amount,
            "hour": hour,
            "is_electronics": is_electronics,
            "tx_per_day": tx_per_day
        }

        # 2. Zapytanie do API
        response = requests.post(API_URL, json=features)
        result = response.json()

        # 3. Jeśli fraud → alert
        if result.get("is_fraud"):
            alert = {
                "timestamp": datetime.utcnow().isoformat(),
                "transaction": tx,
                "fraud_probability": result.get("fraud_probability")
            }

            alert_producer.send("alerts", alert)
            print("🚨 ALERT:", alert)

        else:
            print("OK:", result.get("fraud_probability"))

    except Exception as e:
        print("Błąd przetwarzania:", e)

Writing ml_consumer.py


### Zadanie 3.2 — Uruchom pipeline

W 3 terminalach:
1. `python producer.py` (z Ćw. 1)
2. `uvicorn fraud_api:app --host 0.0.0.0 --port 8001`
3. `python ml_consumer.py`

Obserwuj alerty.

---

## Praca domowa

1. Porównaj wyniki scoringu regułowego (Ćw. 1) vs ML — który lepiej wykrywa?
2. Dodaj endpoint `GET /health` do API.
3. Wypchnij do Git.

**Następne zajęcia:** Apache Spark — przetwarzanie na dużą skalę.